<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

In [2]:
scl_mapping = {
    0: 'no_data',
    1: 'saturated_or_defective_pixel',
    2: 'topographic_casted_shadows',
    3: 'cloud_shadows',
    4: 'vegetation',
    5: 'not_vegetated',
    6: 'water',
    7: 'unclassified',
    8: 'cloud_medium_probability',
    9: 'cloud_high_probability',
    10: 'thin_cirrus',
    11: 'snow_or_ice'
}
scl_mapping

{0: 'no_data',
 1: 'saturated_or_defective_pixel',
 2: 'topographic_casted_shadows',
 3: 'cloud_shadows',
 4: 'vegetation',
 5: 'not_vegetated',
 6: 'water',
 7: 'unclassified',
 8: 'cloud_medium_probability',
 9: 'cloud_high_probability',
 10: 'thin_cirrus',
 11: 'snow_or_ice'}

In [17]:
solar_stat = "Max"

In [18]:
to_drop = ["Latitude", "Longitude", "datetime"]
target = "UHI Index"

df = pd.concat([
    pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
    pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
    pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv"),
], axis=1).drop(to_drop, axis=1, errors="ignore")
feature_shape = df.shape[1]
feature_shape

191

In [27]:
from sklearn.feature_selection import SelectPercentile, f_regression, mutual_info_regression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    VotingRegressor,
    StackingRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    AdaBoostRegressor
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

def calculate_vif_(X, thresh=5.0):
    print("Detecting multicolinearity...")
    X = X.assign(const=1)  # faster than add_constant from statsmodels
    variables = list(range(X.shape[1]))
    dropped = True
    while dropped:
        dropped = False
        vif = [variance_inflation_factor(X.iloc[:, variables].values, ix)
               for ix in range(X.iloc[:, variables].shape[1])]
        vif = vif[:-1]  # don't let the constant be removed in the loop.
        maxloc = vif.index(max(vif))
        if max(vif) > thresh:
            print('dropping \'' + X.iloc[:, variables].columns[maxloc] +
                  '\' at index: ' + str(maxloc))
            del variables[maxloc]
            dropped = True

    print('Remaining variables:')
    print(X.columns[variables[:-1]])
    return X.iloc[:, variables[:-1]]

def add_features(df):
    # Existing features
    stats = ["median", "mean", "min", "max", "var", "std"]
    epsilon = 1e-7

    count_cols = df.columns[df.columns.str.contains("count")]
    for col in count_cols:
        divider = int(col.split("_")[0].replace("m", "")[:-1])
        df[f"{col}_density_per_{divider}m"] = df[col] / divider

    for stat in stats:
        df[f"{stat}_evi_x_lwir"] = df[f"evi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_x_lwir"] = df[f"ndbi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_/_bldg_dnsty"] = df[f"ndbi_{stat}"] / df[f"building_density"].add(epsilon)
        df[f"{stat}_ndbi_/_ndwi"] = df[f"ndbi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_ndvi"] = df[f"ndbi_{stat}"] / df[f"ndvi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_evi"] = df[f"ndbi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_wvp_/_lwir"] = df[f"wvp_{stat}"] / df[f"lwir_{stat}"].add(epsilon)
        df[f"{stat}_infra_red_combo"] = df[f"nir08_{stat}"] * df[f"swir16_{stat}"] * df[f"swir22_{stat}"]
        df[f"{stat}_relative_ndvi"] = df[f"ndvi_{stat}"] / (df[f"ndvi_{stat}"].max() + epsilon)
        df[f"{stat}_relative_ndwi"] = df[f"ndwi_{stat}"] / (df[f"ndwi_{stat}"].max() + epsilon)
        df[f"{stat}_ndvi_ndwi_ratio"] = df[f"ndvi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_evi_ratio"] = df[f"ndvi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndwi_evi_ratio"] = df[f"ndwi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_ndwi_diff"] = df[f"ndvi_{stat}"] - df[f"ndwi_{stat}"]
        df[f"{stat}_ndvi_evi_diff"] = df[f"ndvi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_ndwi_evi_diff"] = df[f"ndwi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_combined_spectral_index"] = (df[f"ndvi_{stat}"] + df[f"ndwi_{stat}"] + df[f"evi_{stat}"]) / 3

    df["normalized_distance_range"] = df.distance_range / df.average_distance.add(epsilon)
    df["normalized_distance_variation"] = df.distance_variation / df.average_distance.add(epsilon)
    df["distance_building_size_interaction"] = df.nearest_building_distance * df.nearest_building_size
    df["distance_building_density_interaction"] = df.nearest_building_distance * df.building_density
    df["average_distance_squared"] = df.average_distance ** 2
    df["nearest_building_distance_squared"] = df.nearest_building_distance ** 2
    df["log_building_area_density"] = np.log(df.building_area_density.add(epsilon))
    df["log_nearest_building_distance"] = np.log(df.nearest_building_distance.add(epsilon))
    df["distance_std_to_mean_ratio"] = df.average_distance / df.average_distance.add(epsilon)
    df["distance_variation_to_range_ratio"] = df.distance_variation / df.distance_range.add(epsilon)

    # Dangerous BOCOR feature!
    return df

def create_train(df_features, target="UHI Index", train_size=0.8):
    # print("Removing duplicates...")
    # rows_before = df_features.shape[0]
    # check_dupl = df_features.columns[1:]
    # df_features = df_features.drop_duplicates(subset=check_dupl, keep='first')
    # rows_after = df_features.shape[0]
    # print(f"Removed {rows_before-rows_after} duplicate rows!")

    # ########################################
    # global index_features, building_features
    # experiment = np.concatenate((index_features, building_features))
    # try:
    #     X = df_features.drop(target, axis=1)[experiment]
    # except:
    #     X = df_features.drop(target, axis=1)
    # ########################################
    X = df_features.drop(target, axis=1)
    y = df_features[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)

    return X_train, X_test, y_train, y_test

def round1(models, train_size):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"

    # Round 1 to get pareto + 1 features
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    print(spaces, "Starting round 1", spaces)
    print(separator)
    global solar_stat
    df = pd.concat([
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv")
    ], axis=1).drop(to_drop, axis=1, errors="ignore")

    building_features = df.iloc[:, -23:].columns
    df = df.pipe(add_features)

    X_train, X_test, y_train, y_test = create_train(df, train_size=train_size)

    # # # # # # # # # # #
    select = SelectPercentile(mutual_info_regression, percentile=25)
    select.fit(X_train, y_train)
    X_train = X_train[select.get_feature_names_out()]
    X_test = X_test[X_train.columns]
    # # # # # # # # # # #

    for model in tqdm(models):
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results)
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    if isinstance(best_model.model, (VotingRegressor, StackingRegressor)):
        print("Best model is either voting or stacking!")
        return
    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return importance

def round2(models, pareto_threshold, importance, train_size):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    # Round 2 to get pareto + 1
    pareto = importance[importance.cumulative_importance <= pareto_threshold]

    print(equals, "Pareto Features + 1", equals)
    print(pareto)
    print(separator)

    print(spaces, "Starting round 2", spaces)
    print(separator)
    global solar_stat
    df = pd.concat([
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
        pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv")
    ], axis=1)
    df = df.pipe(add_features)

    use_cols = [target, *pareto.Features]
    df = df.loc[:, use_cols]
    X_train, X_test, y_train, y_test = create_train(df, train_size=train_size)
    n_features = int(X_train.shape[1] // 3)

    ######################################################
    # X_train = pd.concat([X_train, X_test])
    # y_train = pd.concat([y_train, y_test])
    ######################################################

    for model in tqdm(models):
        model["model"].set_params(max_features=n_features)
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results.head(1))
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return best_model, X_train

# Modelling

In [31]:
isinstance(1.0, (int, float))

True

In [28]:
max_features = int(feature_shape // 3)
basic_params = {
    "n_jobs": -1,
    "random_state": 0,
}

vote = VotingRegressor(
    [
        ("1", ExtraTreesRegressor(max_features=0.25, n_estimators=200, **basic_params)),
        ("2", ExtraTreesRegressor(max_features=0.5, n_estimators=200, **basic_params)),
        ("3", ExtraTreesRegressor(max_features=0.75, n_estimators=200, **basic_params)),
        ("4", ExtraTreesRegressor(max_features=1.0, n_estimators=200, **basic_params)),
        ("5", ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params)),
    ],
    n_jobs=-1
)

stack = StackingRegressor(
    [
        ("1", ExtraTreesRegressor(max_features=0.25, n_estimators=200, **basic_params)),
        ("2", ExtraTreesRegressor(max_features=0.5, n_estimators=200, **basic_params)),
        ("3", ExtraTreesRegressor(max_features=0.75, n_estimators=200, **basic_params)),
        ("4", ExtraTreesRegressor(max_features=1.0, n_estimators=200, **basic_params)),
    ],
    ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params),
    cv=2,
    n_jobs=-1
)

models = [
    # {"model": RandomForestRegressor(**basic_params, max_features=max_features)},
    # {"model": RandomForestRegressor(250, max_features=max_features, **basic_params)},
    # {"model": RandomForestRegressor(150, max_features=max_features, **basic_params)},
    {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=250, **basic_params)},
    {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=150, **basic_params)},
    # {"model": GradientBoostingRegressor(max_features=max_features, n_estimators=250, random_state=0)},
    # {"model": BaggingRegressor(max_features=max_features, n_estimators=250, **basic_params)},
    # {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=100, **basic_params)},
    # {"model": ExtraTreesRegressor(max_features=max_features, n_estimators=300, **basic_params)},
    # {"model": vote},
    # {"model": stack},
]

importance = round1(models, train_size=0.95)

# Best score thrshld 0.75
# 0.974421

                         Starting round 1                         
------------------------------------------------------------------


100%|██████████| 2/2 [01:00<00:00, 30.31s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features=63, random_st...       1.0   0.974696
1  (ExtraTreeRegressor(max_features=63, random_st...       1.0   0.974300
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features=63, n_estimators=250, n_jobs=-1,
                    random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                    Features  Importance  cumulative_importance
0           average_distance    0.059670                   0.06
1             distance_range    0.058136                   0.12
2   average_distance_squared    0.057099                   0.17
3         distance_variation    0.050865     

In [32]:
best_model, X_train = round2(models, pareto_threshold=0.5, importance=importance,
                             train_size=0.95)
# pareto=0.5, split=0.95 | 0.979597 # solar min # SCORE 0.9788
# pareto=0.8, split=0.95 | 0.974361 # solar min # SCORE 0.9758
# pareto=0.8, split=0.8 | 0.973956 # solar min
# pareto=0.5, split=0.95 | 0.979852 # solar max # SCORE 0.9797
# pareto=0.55, split=0.95 | 0.979852 # solar max # SCORE 0.979
# pareto=0.55, split=0.8 | 0.977282 # solar max
# pareto=0.5, split=0.95 | 0.98004 # solar max Feature MI # SCORE 0.9804

================================ Pareto Features + 1 ================================
                    Features  Importance  cumulative_importance
0           average_distance    0.059670                   0.06
1             distance_range    0.058136                   0.12
2   average_distance_squared    0.057099                   0.17
3         distance_variation    0.050865                   0.23
4            median_distance    0.046912                   0.27
5               max_distance    0.036945                   0.31
6                  atran_min    0.027008                   0.34
7          coast_aerosol_max    0.026627                   0.36
8         apparent_elevation    0.026596                   0.39
9           airmass_relative    0.026361                   0.42
10                 lwir_mean    0.023252                   0.44
11                    zenith    0.022941                   0.46
12           apparent_zenith    0.021933                   0.48
13                

100%|██████████| 2/2 [00:06<00:00,  3.33s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features=4, random_sta...       1.0    0.98004
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features=4, n_estimators=250, n_jobs=-1, random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                    Features  Importance  cumulative_importance
0            apparent_zenith    0.078874                   0.08
1               max_distance    0.078542                   0.16
2                  elevation    0.078226                   0.24
3             distance_range    0.077126                   0.31
4   average_distance_squared    0.074545                   0.39
5           

# Predicting Submission

In [33]:
def create_submission(filename: str, model):
    global solar_stat
    sub_df = pd.concat([
        pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Final.csv"),
        pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/Submission_SolarData{solar_stat}SinceTrainTime.csv"),
    ], axis=1)

    final_df = sub_df[["Latitude", "Longitude"]].copy()
    print("Predicting", sub_df.shape[0], "rows...")

    ############################
    sub_df = add_features(sub_df)

    # # # # # Comment if not used!
    # sub_df.scl_median = sub_df.scl_median.map(scl_mapping)
    # scl_ohe = ohe.transform(sub_df.loc[:, ["scl_median"]])
    # scl_ohe = pd.DataFrame(scl_ohe, columns=ohe.get_feature_names_out(["scl_median"]))
    # sub_df = pd.concat([sub_df.drop("scl_median", axis=1), scl_ohe], axis=1)

    to_predict = sub_df.loc[:, X_train.columns]

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return
create_submission(f"DynamicMaxFeatures_SolarStat{solar_stat}_FeatureMI.csv", best_model.model)

Predicting 1040 rows...
Predicting...
Done!


---

# PLAYGROUND

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

sub_df = pd.read_csv("https://challenge.ey.com/api/v1/storage/admin-files/8403843075872017-678a346b124df4f88e67ef26-Submission_template_UHI2025-v2.csv")
df = pd.read_csv("https://challenge.ey.com/api/v1/storage/admin-files/7286921994693265-678a3479124df4f88e67ef7e-Training_data_uhi_index_UHI2025-v2.csv")

In [ ]:
target = "UHI Index"
to_drop = ["Latitude", "Longitude", "datetime", target]
train_build = pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv")
df_train = pd.concat([df, train_build], axis=1)

X = df_train.drop(to_drop, axis=1, errors="ignore")
y = df_train[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=0.8)

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.ensemble import ExtraTreesRegressor
selector = RFECV(ExtraTreesRegressor(), step=1, cv=5, n_jobs=-1)
selector.fit(X, y)
selected_features = selector.support_

In [ ]:
# from sklearn.feature_selection import SequentialFeatureSelector

# sfs = SequentialFeatureSelector(
#     ExtraTreesRegressor(random_state=0, n_jobs=-1),
#     n_features_to_select="auto",
#     direction="forward",
#     # scoring: Any | None = None,
#     cv=2,
#     n_jobs=-1
# )
# sfs.fit(X_train, y_train)

SequentialFeatureSelector(cv=2,
                          estimator=ExtraTreesRegressor(n_jobs=-1,
                                                        random_state=0),
                          n_jobs=-1)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif_(X, thresh=5.0):
    X = X.assign(const=1)  # faster than add_constant from statsmodels
    variables = list(range(X.shape[1]))
    dropped = True
    while dropped:
        dropped = False
        vif = [variance_inflation_factor(X.iloc[:, variables].values, ix)
               for ix in range(X.iloc[:, variables].shape[1])]
        vif = vif[:-1]  # don't let the constant be removed in the loop.
        maxloc = vif.index(max(vif))
        if max(vif) > thresh:
            print('dropping \'' + X.iloc[:, variables].columns[maxloc] +
                  '\' at index: ' + str(maxloc))
            del variables[maxloc]
            dropped = True

    print('Remaining variables:')
    print(X.columns[variables[:-1]])
    return X.iloc[:, variables[:-1]]
# X_train = calculate_vif_(X_train, thresh=3.5)
# X_test = X_test[X_train.columns]

dropping '40m_nearby_building_count' at index: 4
Remaining variables:
Index(['is_a_building', '10m_nearby_building_count',
       '20m_nearby_building_count', '30m_nearby_building_count',
       '60m_nearby_building_count', 'average_distance', 'building_density',
       'building_area_density', 'neighboring_intersection',
       'relative_position', 'nearest_polygon_angle', 'avg_polygon_complexity'],
      dtype='object')


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(150, random_state=69, n_jobs=-1)
et.fit(X_train, y_train)
print("Insample  :", r2_score(y_train, et.predict(X_train)))
print("Outsample :", r2_score(y_test, et.predict(X_test)))

In [ ]:
importance = pd.DataFrame({"Features": X_train.columns, "Importance": et.feature_importances_}).sort_values("Importance", ascending=False).reset_index(drop=True)
importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
importance

In [ ]:
threshold = 1.0
pareto = importance[importance.cumulative_importance <= threshold].Features
pareto

,Features
0,distance_range
1,average_distance
2,std_distance
3,distance_variation
4,building_area_density
5,avg_polygon_complexity
6,neighboring_intersection
7,nearest_building_size
8,nearest_polygon_angle
9,is_a_building


In [ ]:
et.fit(X_train[pareto], y_train)
print("Insample  :", r2_score(y_train, et.predict(X_train[pareto])))
print("Outsample :", r2_score(y_test, et.predict(X_test[pareto])))

Insample  : 1.0
Outsample : 0.951773657537061


In [ ]:
sub_build = pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Building_Features.csv")

In [ ]:
# sub_df["UHI Index"] = et.predict(sub_build)
# sub_df.to_csv("PlaygroundAe.csv", index=False)

sub_df["UHI Index"] = et.predict(sub_build[pareto])
sub_df.to_csv("PlaygroundRFECV.csv", index=False)